# JAX and Keras on TPU — Reference Sheet

Runnable companion to the [PyTorch, JAX, Keras, NumPy, and Python reference sheet](https://raila.io/#/blog/pytorch-numpy-python-reference) on [raila.io](https://raila.io).

**Before running:** in the menu above, go to `Runtime` -> `Change runtime type` -> set `Hardware accelerator` to `TPU`, then `Save`. The first cell below checks that it took effect.

## 1. Confirm the TPU is visible to JAX

In [ ]:
import jax

print(jax.devices())
# On a TPU runtime this should list TPU devices, e.g. [TpuDevice(id=0, ...), ...].
# If it prints CpuDevice instead, double check Runtime > Change runtime type > TPU, then rerun.

## 2. `jax.numpy` basics

In [ ]:
import jax.numpy as jnp

x = jnp.arange(12).reshape(3, 4)
print(x)
print(x.sum(axis=0))
print(x.dtype, x.device)

## 3. Arrays are immutable — use `.at[...].set(...)`

In [ ]:
x = jnp.zeros(5)
y = x.at[2].set(9.0)  # returns a *new* array; x is unchanged
print('x:', x)
print('y:', y)

## 4. Functional transforms: `jit`, `grad`, `vmap`

In [ ]:
def f(v):
    return jnp.sum(v ** 2)

grad_f = jax.grad(f)
fast_f = jax.jit(f)
batched_f = jax.vmap(f)

v = jnp.array([1.0, 2.0, 3.0])
print("f(v)      =", fast_f(v))
print("grad f(v) =", grad_f(v))
print("batched   =", batched_f(jnp.stack([v, v * 2])))

## 5. Explicit PRNG keys

In [ ]:
key = jax.random.key(0)
key, subkey = jax.random.split(key)
sample = jax.random.normal(subkey, (2, 3))
print(sample)

## 6. Keras 3 on the JAX backend, running on TPU

`KERAS_BACKEND` must be set *before* `import keras` — restart the runtime if you change it after Keras has already been imported once in this session.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import keras
from keras import layers

print("Keras backend:", keras.backend.backend())

Build a small MLP and train it on synthetic data — swap in a real dataset as needed.

In [ ]:
import numpy as np

num_samples, num_features, num_classes = 2048, 20, 4
rng = np.random.default_rng(0)
x_train = rng.normal(size=(num_samples, num_features)).astype('float32')
y_train = rng.integers(0, num_classes, size=(num_samples,))

model = keras.Sequential([
    layers.Input(shape=(num_features,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(num_classes),
])

model.compile(
    optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

model.fit(x_train, y_train, batch_size=64, epochs=5, validation_split=0.1)